# Qwen3.8-27B on BFCL v4 (thinking on)

`Qwen/Qwen3.8-27B` used as is, with no fine-tuning, on the **Berkeley Function Calling Leaderboard (BFCL v4)**: every category except
web search (it needs a paid SerpAPI key) and format sensitivity (prompt-mode models only). That is about **4,900 test entries**:
non-live and live single-turn calls, multi-turn conversations with stateful tools, and the three memory backends.

The model runs in **FC mode** (native tool calls, parsed by vLLM's `qwen3_coder` parser) with **thinking on**, temperature 0.001,
32 requests at a time, served by vLLM 0.30.0 in bf16 on one A100 80GB. BFCL scores with its own checkers (AST matching, state
checks, answer matching), so no LLM judge is involved.

Data: `results/bfcl_run.json`, written every 30 s by `update_json.py` during the run (`./updater.sh start`).

In [1]:
import json, math, datetime
from pathlib import Path
from collections import Counter
import numpy as np
from IPython.display import HTML, display
P = next(p for p in (Path("results/bfcl_run.json"), Path("../results/bfcl_run.json")) if p.exists())
GROUP_COL = {"non-live": "#2F5D8A", "live": "#1E7A5C", "multi-turn": "#7A4E9C", "memory": "#E0883B"}
C_MAIN = "#7A4E9C"
def fmt_min(m):
    if m is None: return "-"
    m = int(round(m)); return f"{m} min" if m < 60 else f"{m // 60} h {m % 60:02d} min"
def hm(iso):   # "2026-10-11T08:09:41" -> "8:09" (24-hour, no leading zero)
    if not iso: return "-"
    t = datetime.datetime.fromisoformat(iso); return f"{t.hour}:{t.minute:02d}"
def table(rows, head=None):
    h = "".join(f"<th style='text-align:left;padding:2px 10px'>{x}</th>" for x in head) if head else ""
    b = "".join("<tr>" + "".join(f"<td style='padding:2px 10px'>{x}</td>" for x in r) + "</tr>" for r in rows)
    return f"<table style='border-collapse:collapse;font-size:13px'>{'<tr>' + h + '</tr>' if h else ''}{b}</table>"
def load():
    R = json.load(open(P))
    group_of = {c: g for g, cs in R["groups"].items() for c in cs}
    return R, group_of
def acc(c):   # (accuracy, binomial s.e., n) from BFCL's score file, or None if not evaluated yet
    s = c.get("score")
    if not s or not s.get("total"): return None
    p, n = s["accuracy"], s["total"]; return p, math.sqrt(p * (1 - p) / n), n
R, GROUP_OF = load()
print(f"stage: {R['stage']} | {R['plan']['done']}/{R['plan']['total']} entries | JSON updated {R['updated']}")

stage: finished | 4906/4906 entries | JSON updated 2026-10-12T03:19:56


## 1. Run info and estimated time

In [2]:
def info_html(R):
    pl, pod, cfg, cats = R["plan"], R.get("pod", {}), R["config"], R["categories"]
    o = [f"<h4>JSON written {R['updated']} &middot; stage: <span style='color:{C_MAIN}'>{R['stage']}</span></h4>"]
    if not R.get("pod_reachable", True): o.append("<p style='color:#B3261E'><b>Pod unreachable</b>, showing the last known state.</p>")
    scored = [acc(c) for c in cats.values() if acc(c)]
    avg = sum(a[0] for a in scored) / len(scored) if scored else None
    official = (R.get("csv", {}).get("data_overall") or {}).get("Overall Acc")
    st = (pod.get("vllm_stats") or {})
    o.append("<b>Progress</b>" + table([
        ["entries done", f"{pl['done']} / {pl['total']} ({pl['done'] / pl['total']:.0%})"],
        ["categories scored", f"{len(scored)} / {len(cats)}"],
        ["BFCL overall accuracy", f"<b>{official}</b> (official weighting)" if official and R['stage'] == 'finished' else
            (f"{avg:.1%} plain average of the {len(scored)} scored categories (not BFCL's weighting)" if avg is not None else "-")],
        ["started", hm(pl.get("started_at"))], ["elapsed", fmt_min(pl["elapsed_h"] * 60) if pl.get("elapsed_h") else "-"],
        ["time left (rough)", fmt_min(pl.get("left_min")) if pl.get("left_min") else ("done" if pl["done"] >= pl["total"] else "measuring...")],
        ["estimated finish (local)", f"<b>{pl.get('done_at') or '-'}</b>"],
        ["BFCL's own estimate (assumes all entries are equally slow)", f"{fmt_min(pl['bfcl_eta']['left_min'])} left, done ≈ {pl['bfcl_eta']['done_at']} ({pl['bfcl_eta']['done']}/{pl['bfcl_eta']['total']} in this pass)" if pl.get("bfcl_eta") else "-"],
        ["pod cost so far / projected", f"${pl['cost_so_far']:.2f} / ${pl['cost_projected']:.2f}" if pl.get("cost_projected") else "-"],
        ["vLLM now", f"{st.get('running', 0):.0f} running, {st.get('waiting', 0):.0f} waiting, {st.get('gen_tps', 0):.0f} generated tokens/s" if st else "-"]]))
    rows = []
    for g, v in pl["groups"].items():
        rows.append([f"<span style='color:{GROUP_COL[g]}'>&#9632;</span> {g}", f"{v['done']} / {v['total']}",
                     f"{v['min_per_entry'] * 60:.1f} s" if v.get("min_per_entry") else "-", "measured" if v.get("measured") else "guess",
                     fmt_min(v["minutes"]) if v.get("minutes") else "-"])
    o.append(f"<b>Each group</b> (wall-clock seconds per entry, measured since the last restart, with {cfg.get('threads', '?')} running at once)" + table(rows, ["group", "entries", "s / entry", "", "time spent"]))
    o.append("<b>Settings</b>" + table([[k.replace("_", " "), v] for k, v in cfg.items()]))
    o.append("<b>Machine</b>" + table([["GPU (name, load, memory used, total)", pod.get("gpu") or "-"],
        ["disk / RAM disk", f"{pod.get('disk') or '-'} &middot; {pod.get('shm') or '-'}"], ["vLLM answering", "yes" if pod.get("vllm_up") else "no"],
        ["processes alive", ", ".join(k for k, v in (pod.get("procs") or {}).items() if v) or "none"],
        ["last BFCL log line", (pod.get("bfcl_log_tail") or "-").replace("<", "&lt;")[-150:]]]))
    o.append("<b>Recent events</b>" + table([[hm(e["time"]), e["event"].replace("<", "&lt;")] for e in R.get("events", [])[-8:]]))
    return "".join(o)
display(HTML(info_html(R)))

In [3]:
# live view: refreshes every 30 s until the run finishes (interrupt the kernel to stop)
import time
from IPython.display import clear_output
try:
    while True:
        R, GROUP_OF = load(); clear_output(wait=True); display(HTML(info_html(R)))
        if str(R["stage"]).startswith(("finished", "failed")): break
        time.sleep(30)
except KeyboardInterrupt:
    pass

---
# Graphs

In [4]:
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
pio.renderers.default = "notebook_connected"; pio.templates.default = "plotly_white"
def placeholder(title, text="no data yet"):
    f = go.Figure(); f.add_annotation(text=text, showarrow=False, font=dict(size=16, color="#888"))
    f.update_layout(title=title, height=240, xaxis=dict(visible=False), yaxis=dict(visible=False)); f.show()

In [5]:
R, GROUP_OF = load()
CATS = R["categories"]; ORDER = [c for g in R["groups"].values() for c in g]
SCORED = [c for c in ORDER if acc(CATS[c])]
LABEL = {c: c.replace("_", " ") for c in ORDER}
def group_acc(g):   # plain average of the group's scored categories (BFCL averages categories, not entries)
    v = [acc(CATS[c])[0] for c in R["groups"][g] if acc(CATS[c])]
    return sum(v) / len(v) if v else None
FAILS = {c: CATS[c].get("fail_types", {}) for c in SCORED}
ERR_NAMES = [   # BFCL error_type (prefix:suffix) -> readable label; first match wins
    (":wrong_count", "wrong number of calls"), (":wrong_func_name", "wrong function"), ("missing_required", "missing required argument"),
    ("unexpected_param", "extra argument"), ("missing_optional", "missing optional argument"), ("value_error:string", "wrong value (text)"),
    ("value_error:list", "wrong value (list)"), ("value_error:dict", "wrong value (dict)"), ("value_error", "wrong value (number/other)"),
    ("type_error", "wrong argument type"), ("decoder_failed", "output not parseable"), ("irrelevance", "called a tool it shouldn't"),
    ("relevance", "no call when one was needed"), ("instance_state_mismatch", "wrong final state"), ("execution_response_mismatch", "wrong results returned"),
    ("empty_turn", "answered nothing in a turn"), ("force_terminated", "too many steps in a turn"), ("answer_not_found", "answer not found"),
]
def short_err(e):
    for key, name in ERR_NAMES:
        if key in e: return name
    return e.replace(":", " / ").replace("_", " ")
print(f"{len(SCORED)} of {len(ORDER)} categories scored so far")

20 of 20 categories scored so far


## 2. Progress per category

In [6]:
fig = go.Figure()
cs = ORDER[::-1]
fig.add_bar(y=[LABEL[c] for c in cs], x=[CATS[c]["total"] for c in cs], orientation="h", marker_color="#E6E6E6", name="total", hoverinfo="skip")
fig.add_bar(y=[LABEL[c] for c in cs], x=[min(CATS[c]["done"], CATS[c]["total"]) for c in cs], orientation="h", marker_color=[GROUP_COL[GROUP_OF[c]] for c in cs], name="done",
            text=[f"{CATS[c]['done']}/{CATS[c]['total']}" for c in cs], textposition="outside")
fig.update_layout(title="Entries generated per category (grey = still to do)", barmode="overlay", height=620, showlegend=False, xaxis_title="entries")
fig.show()

## 3. Accuracy per category and per group
BFCL averages categories (not entries), so a small category like live_parallel (16 entries) counts as much as live_multiple (1,053). Error bars are binomial standard errors.

In [7]:
if SCORED:
    cs = SCORED[::-1]; a = [acc(CATS[c]) for c in cs]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], subplot_titles=("Accuracy per category (±1 s.e.)", "Group averages (scored categories only)"))
    fig.add_bar(y=[f"{LABEL[c]} (n={x[2]})" for c, x in zip(cs, a)], x=[x[0] for x in a], error_x=dict(type="data", array=[x[1] for x in a]), orientation="h",
                marker_color=[GROUP_COL[GROUP_OF[c]] for c in cs], text=[f"{x[0]:.1%}" for x in a], textposition="outside", showlegend=False, row=1, col=1)
    gs = [g for g in R["groups"] if group_acc(g) is not None]
    fig.add_bar(x=gs, y=[group_acc(g) for g in gs], marker_color=[GROUP_COL[g] for g in gs], text=[f"{group_acc(g):.1%}" for g in gs], textposition="outside", showlegend=False, row=1, col=2)
    fig.update_xaxes(range=[0, 1.12], tickformat=".0%", row=1, col=1); fig.update_yaxes(range=[0, 1.1], tickformat=".0%", row=1, col=2)
    fig.update_layout(height=max(420, 26 * len(cs) + 120)); fig.show()
else:
    placeholder("Accuracy", "no category scored yet")

## 4. Why entries failed
BFCL's own error types from the score files: wrong number of calls, wrong argument values, a call where none was needed, a missing state change in multi-turn, and so on.

In [8]:
if any(FAILS.values()):
    errs = Counter(); [errs.update({short_err(k): v}) for f in FAILS.values() for k, v in f.items()]
    top = [e for e, _ in errs.most_common(8)]
    cs = [c for c in SCORED if FAILS[c]][::-1]; fig = go.Figure()
    palette = ["#B3261E", "#E0883B", "#E8B04A", "#9DB8D9", "#2F5D8A", "#7A4E9C", "#1E7A5C", "#6B6B6B", "#CFCFCF"]
    for i, e in enumerate(top + ["other"]):
        vals = [sum(v for k, v in FAILS[c].items() if (short_err(k) == e) or (e == "other" and short_err(k) not in top)) for c in cs]
        if any(vals): fig.add_bar(y=[LABEL[c] for c in cs], x=vals, orientation="h", name=e, marker_color=palette[i % len(palette)])
    fig.update_layout(title="Why entries failed (BFCL error types)", barmode="stack", height=max(380, 26 * len(cs) + 160), xaxis_title="failed entries", legend=dict(orientation="h", y=-0.15))
    fig.show()
    print("most common:", ", ".join(f"{e} {n}" for e, n in errs.most_common(8)))
else:
    placeholder("Failures", "no failures recorded yet")

most common: called a tool it shouldn't 272, wrong final state 199, wrong value (text) 149, answer not found 127, wrong value (number/other) 63, wrong number of calls 61, answered nothing in a turn 56, wrong results returned 42


## 5. How much it thinks, and roughly how long each category took
All categories ran in one overlapping pass, so there is no exact per-category timer. Real time per phase comes from the pod log (single-turn until its last category was scored, then multi-turn, then memory), split inside each phase by GPU work: output tokens + 0.1 × input tokens. Treat the bars as estimates.

In [9]:
cs = [c for c in ORDER if CATS[c].get("out_tokens_each")]
if cs:
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], subplot_titles=("Output tokens per entry, thinking included (log scale)", "Approximate time per category (minutes)"))
    for c in cs:
        fig.add_box(x=CATS[c]["out_tokens_each"], name=LABEL[c], marker_color=GROUP_COL[GROUP_OF[c]], boxmean=True, showlegend=False, orientation="h", row=1, col=1)
    tm = [c for c in ORDER if CATS[c].get("approx_min") is not None]
    fig.add_bar(y=[LABEL[c] for c in tm], x=[CATS[c]["approx_min"] for c in tm], orientation="h", marker_color=[GROUP_COL[GROUP_OF[c]] for c in tm],
                text=[fmt_min(CATS[c]["approx_min"]) for c in tm], textposition="outside", showlegend=False, row=1, col=2)
    fig.update_xaxes(type="log", row=1, col=1); fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=max(420, 26 * len(cs) + 120)); fig.show()
    tot_in = sum(CATS[c].get("input_tokens", 0) for c in ORDER); tot_out = sum(CATS[c].get("output_tokens", 0) for c in ORDER)
    print(f"tokens so far: {tot_in / 1e6:.1f} M in, {tot_out / 1e6:.2f} M out")
else:
    placeholder("Tokens", "no entries yet")

tokens so far: 88.0 M in, 4.54 M out


## 6. All categories in one table

In [9]:
rows = []
for c in ORDER:
    x, cc = acc(CATS[c]), CATS[c]
    rows.append([f"<span style='color:{GROUP_COL[GROUP_OF[c]]}'>&#9632;</span> {GROUP_OF[c]}", c, f"{cc['done']}/{cc['total']}",
                 f"{x[0]:.1%}" if x else "-", f"{cc['score']['correct']}/{cc['score']['total']}" if x else "-",
                 fmt_min(cc.get("approx_min")) if cc.get("approx_min") else "-", hm(cc.get("finished_at")), f"{cc.get('n_errors', 0)}"])
display(HTML(table(rows, ["group", "category", "generated", "accuracy", "correct", "≈ time", "finished at", "inference errors"])))
csv = R.get("csv", {}).get("data_overall")
if csv:
    keep = [k for k in csv if k.endswith("Acc") or k in ("Overall Acc", "Latency Mean (s)")]
    display(HTML("<b>BFCL's own summary (data_overall.csv)</b>" + table([[k, csv[k]] for k in keep])))

group,category,generated,accuracy,correct,≈ time,finished at,inference errors
■ non-live,simple_python,400/400,92.5%,370/400,9 min,12:30,0
■ non-live,simple_java,100/100,59.0%,59/100,8 min,15:15,0
■ non-live,simple_javascript,50/50,70.0%,35/50,6 min,15:19,1
■ non-live,parallel,200/200,92.0%,184/200,8 min,14:23,0
■ non-live,multiple,200/200,92.5%,185/200,5 min,14:14,0
■ non-live,parallel_multiple,200/200,85.0%,170/200,11 min,14:33,0
■ non-live,irrelevance,240/240,79.2%,190/240,17 min,12:51,0
■ live,live_simple,258/258,86.4%,223/258,8 min,14:16,0
■ live,live_multiple,1053/1053,79.1%,833/1053,41 min,14:10,1
■ live,live_parallel,16/16,81.2%,13/16,1 min,14:03,0


Overall Acc,56.86%
Latency Mean (s),https://huggingface.co/Qwen/Qwen3.8-27B
Non-Live AST Acc,0.02
Live Acc,92.00%
Multi Turn Acc,81.25%
Web Search Acc,51.50%
Memory Acc,N/A
